In [2]:
!pip install pyspark

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, sum, when, round
import pyspark.sql.functions as F

# Supply Chain SLA Analysis — PySpark Extension
# ---
# Started this as a Pandas/SQL project but wanted to see if the same
# logic could scale. Rewrote the core pipeline in PySpark so it can
# handle enterprise-level volumes without hitting memory limits.
# Same 5 stages, same dataset, distributed this time.

spark = SparkSession.builder \
    .appName("SLA Analysis") \
    .getOrCreate()

# Load data
# Had the same encoding issue as the original script — ISO-8859-1
# was the fix for the city/region columns. Dropping rows where
# shipping dates are null because the SLA math doesn't work without both.

df = spark.read.csv(
    "DataCoSupplyChainDataset.csv",
    header=True,
    inferSchema=True,
    encoding="ISO-8859-1"
)

df = df.dropna(subset=[
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Customer Id"
])

print(f"Records loaded: {df.count()}")

Records loaded: 180519


In [7]:
# Stage 1: Integrity check
# Always run this first before touching any business metrics.
# Wanted to make sure there weren't zero-value orders or duplicate
# Order IDs messing up the SLA calculations downstream.

print("\n--- [STAGE 1] DATA INTEGRITY AUDIT ---")

df.agg(
    count("*").alias("total_rows"),
    F.countDistinct("Order Id").alias("unique_orders"),
    sum(when(col("Order Item Total") <= 0, 1).otherwise(0)).alias("pricing_anomalies"),
    sum(when(col("Days for shipping (real)") > 10, 1).otherwise(0)).alias("extreme_delays")
).show()


--- [STAGE 1] DATA INTEGRITY AUDIT ---
+----------+-------------+-----------------+--------------+
|total_rows|unique_orders|pricing_anomalies|extreme_delays|
+----------+-------------+-----------------+--------------+
|    180519|        65752|                0|             0|
+----------+-------------+-----------------+--------------+



In [8]:
# Stage 2: SLA success rates
# Kept the same strict vs buffered split from the SQL version.
# The buffered rate (1-day grace period) was something I added after
# noticing a lot of failures were off by just 1 day — wanted to
# see if it was marginal misses or actual operational gaps.

print("\n--- [STAGE 2] SLA PERFORMANCE: STRICT VS BUFFERED ---")

df.groupBy("Shipping Mode") \
    .agg(
        count("*").alias("order_volume"),
        round(
            avg(when(
                col("Days for shipping (real)") <= col("Days for shipment (scheduled)"), 1.0
            ).otherwise(0.0)) * 100, 2
        ).alias("strict_success_rate_pct"),
        round(
            avg(when(
                col("Days for shipping (real)") <= col("Days for shipment (scheduled)") + 1, 1.0
            ).otherwise(0.0)) * 100, 2
        ).alias("buffered_success_rate_pct")
    ) \
    .orderBy("order_volume", ascending=False) \
    .show()


--- [STAGE 2] SLA PERFORMANCE: STRICT VS BUFFERED ---
+--------------+------------+-----------------------+-------------------------+
| Shipping Mode|order_volume|strict_success_rate_pct|buffered_success_rate_pct|
+--------------+------------+-----------------------+-------------------------+
|Standard Class|      107752|                  60.23|                    79.82|
|  Second Class|       35216|                  20.27|                    40.33|
|   First Class|       27814|                    0.0|                    100.0|
|      Same Day|        9737|                  52.17|                    100.0|
+--------------+------------+-----------------------+-------------------------+



In [9]:
# Stage 3: Latency gap
# This is the one that surfaced the 2-day average delay.
# Grouping by shipping mode to see which tier is leaking the most time.
# Second Class was the biggest problem — promised 2 days, took 4.
# Standard Class was actually perfect. Surprising.

print("\n--- [STAGE 3] LATENCY GAP BY SHIPPING MODE ---")

df.groupBy("Shipping Mode") \
    .agg(
        round(avg("Days for shipment (scheduled)"), 2).alias("promised_days"),
        round(avg("Days for shipping (real)"), 2).alias("actual_days"),
        round(
            avg(
                col("Days for shipping (real)") - col("Days for shipment (scheduled)")
            ), 2
        ).alias("avg_latency_gap_days")
    ) \
    .orderBy("avg_latency_gap_days", ascending=False) \
    .show()


--- [STAGE 3] LATENCY GAP BY SHIPPING MODE ---
+--------------+-------------+-----------+--------------------+
| Shipping Mode|promised_days|actual_days|avg_latency_gap_days|
+--------------+-------------+-----------+--------------------+
|  Second Class|          2.0|       3.99|                1.99|
|   First Class|          1.0|        2.0|                 1.0|
|      Same Day|          0.0|       0.48|                0.48|
|Standard Class|          4.0|        4.0|                 0.0|
+--------------+-------------+-----------+--------------------+



In [10]:
# Stage 4: Customer segmentation
# Wanted to know if the SLA failures were hitting high-spend customers
# more than casual ones. Used the same spend thresholds as the SQL CTE.
# Turns out failure rates were almost identical across all segments —
# ~55% across the board. The problem is systemic, not selective.

print("\n--- [STAGE 4] DELAY IMPACT BY CUSTOMER SEGMENT ---")

df.groupBy("Customer Id") \
    .agg(
        sum("Order Item Total").alias("total_spent"),
        avg("Late_delivery_risk").alias("delay_rate")
    ) \
    .withColumn(
        "customer_segment",
        when(col("total_spent") > 500, "Priority (High Spend)")
        .when(
            (col("total_spent") >= 200) & (col("total_spent") <= 500),
            "Standard (Mid Spend)"
        )
        .otherwise("Casual (Low Spend)")
    ) \
    .groupBy("customer_segment") \
    .agg(
        count("*").alias("user_count"),
        round(avg("delay_rate") * 100, 2).alias("avg_failure_pct")
    ) \
    .orderBy("avg_failure_pct", ascending=False) \
    .show()


--- [STAGE 4] DELAY IMPACT BY CUSTOMER SEGMENT ---
+--------------------+----------+---------------+
|    customer_segment|user_count|avg_failure_pct|
+--------------------+----------+---------------+
|Standard (Mid Spend)|      4230|          55.88|
|  Casual (Low Spend)|      3878|          54.79|
|Priority (High Sp...|     12544|          54.55|
+--------------------+----------+---------------+



In [11]:
# Stage 5: A/B test
# The hypothesis was simple — if we just set more realistic delivery
# estimates (4 days instead of 1), does the success rate recover?
# Split orders into control vs variant using Order ID parity.
# Filtered to First Class only since that's where failures were worst.
# Control: 0% success. Variant: 100% success.
# The network wasn't broken. The promises were.

print("\n--- [STAGE 5] A/B TEST: SERVICE LEVEL RECOVERY ---")

df.filter(col("Shipping Mode") == "First Class") \
    .withColumn(
        "test_group",
        when(col("Order Id") % 2 == 0, "Control (1-Day Promise)")
        .otherwise("Variant (4-Day Estimate)")
    ) \
    .withColumn(
        "success",
        when(
            (col("Order Id") % 2 != 0) &
            (col("Days for shipping (real)") <= 4), 1.0
        ).when(
            (col("Order Id") % 2 == 0) &
            (col("Days for shipping (real)") <= col("Days for shipment (scheduled)")), 1.0
        ).otherwise(0.0)
    ) \
    .groupBy("test_group") \
    .agg(
        round(avg("success") * 100, 2).alias("fulfillment_success_rate_pct")
    ) \
    .show()


--- [STAGE 5] A/B TEST: SERVICE LEVEL RECOVERY ---
+--------------------+----------------------------+
|          test_group|fulfillment_success_rate_pct|
+--------------------+----------------------------+
|Control (1-Day Pr...|                         0.0|
|Variant (4-Day Es...|                       100.0|
+--------------------+----------------------------+

